# 半衰主成分风险平价模型 (HPCRP) 全球资产配置策略研究

## 研报复现 - 天风证券 2017-09-18

本notebook复现天风证券研报《基于半衰主成分风险平价模型的全球资产配置策略研究》中的核心内容。

---

## 1. 环境准备与数据获取

In [ ]:
# 导入必要的库
import sys
import os
import pandas as pd
import numpy as np

# 添加项目路径
sys.path.insert(0, os.getcwd())

from source.data_loader import fetch_global_index_data
print("库导入成功")

In [ ]:
# 获取全球指数数据
returns = fetch_global_index_data()
print(f"\n数据获取完成: {len(returns)}个交易日, {len(returns.columns)}个指数")
print(f"时间范围: {returns.index.min().date()} 至 {returns.index.max().date()}")
print(f"\n指数列表: {list(returns.columns)}")
returns.head()

## 2. 统计性描述

In [ ]:
# 计算各指数的收益风险指标
def calc_stats(returns):
    """计算收益风险统计指标"""
    annual_ret = returns.mean() * 252
    annual_vol = returns.std() * np.sqrt(252)
    max_dd = ((1 + returns).cumprod().cummax() - (1 + returns).cumprod()) / (1 + returns).cumprod().cummax()
    max_dd = max_dd.min()
    sharpe = (annual_ret - 0.03) / annual_vol
    calmar = annual_ret / abs(max_dd)
    
    stats = pd.DataFrame({
        '年化收益率': annual_ret,
        '年化波动率': annual_vol,
        '夏普比率': sharpe,
        '最大回撤': max_dd,
        'Calmar比率': calmar
    })
    return stats

stats = calc_stats(returns)
stats.style.format("{:.2%}")

In [ ]:
# 绘制相关系数矩阵
from source.plot import plot_correlation_matrix
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
corr = returns.corr()
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Correlation')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45)
ax.set_yticklabels(corr.columns)
ax.set_title('Global Index Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. 资产配置模型

In [ ]:
from source.models import (
    get_model_weights,
    equal_weight,
    equal_volatility_weights,
    minimum_variance_weights,
    maximum_diversification_weights,
    risk_parity_weights,
    principal_component_risk_parity_weights,
    hpcrp_weights
)

# 测试各模型权重
print("各模型权重测试 (使用最近240个交易日):\n")

recent = returns.iloc[-240:]
models = ['EW', 'EV', 'MV', 'MD', 'RP', 'PCRP', 'HPCRP']

for model in models:
    if model == 'HPCRP':
        w = hpcrp_weights(recent, half_life=120)
    elif model == 'PCRP':
        w = principal_component_risk_parity_weights(recent)
    elif model == 'EW':
        w = equal_weight(len(recent.columns))
    elif model == 'EV':
        w = equal_volatility_weights(recent)
    elif model == 'MV':
        w = minimum_variance_weights(recent)
    elif model == 'MD':
        w = maximum_diversification_weights(recent)
    elif model == 'RP':
        w = risk_parity_weights(recent)
    
    print(f"{model}:", np.round(w, 3))

## 4. 回测

In [ ]:
from source.backtest import run_backtest, calculate_metrics

# 定义权重函数
def weights_func(model_name, hist_data):
    if model_name == 'HPCRP':
        return get_model_weights(model_name, hist_data, half_life=120)
    return get_model_weights(model_name, hist_data)

# 运行回测
results = {}
models = ['EW', 'EV', 'MV', 'MD', 'RP', 'PCRP', 'HPCRP']

for model in models:
    print(f"运行 {model}...")
    result = run_backtest(
        returns,
        weights_func,
        model,
        rebalance_freq='quarterly',
        window=240,
        start_date='2009-01-01'
    )
    
    # 计算指标
    result['metrics'] = calculate_metrics(result['returns'], result['nav'])
    results[model] = result
    
    m = result['metrics']
    print(f"  年化收益: {m['annual_return']:.2%}, 最大回撤: {m['max_drawdown']:.2%}")

print("\n回测完成!")

## 5. 回测结果

In [ ]:
# 汇总指标
metrics_list = []
for model, result in results.items():
    m = result['metrics'].copy()
    m['model'] = model
    metrics_list.append(m)

metrics_df = pd.DataFrame(metrics_list)
metrics_df = metrics_df[['model', 'annual_return', 'annual_vol', 'sharpe_ratio', 'max_drawdown', 'calmar_ratio']]
metrics_df.columns = ['模型', '年化收益率', '年化波动率', '夏普比率', '最大回撤', 'Calmar比率']
metrics_df = metrics_df.sort_values('Calmar比率', ascending=False)
metrics_df.style.format({
    '年化收益率': '{:.2%}',
    '年化波动率': '{:.2%}',
    '夏普比率': '{:.3f}',
    '最大回撤': '{:.2%}',
    'Calmar比率': '{:.3f}'
})

In [ ]:
# 绘制净值曲线
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']

for i, (model, result) in enumerate(results.items()):
    ax.plot(result['nav'].index, result['nav'].values, label=model, 
           linewidth=1.5, color=colors[i])

ax.set_xlabel('Date')
ax.set_ylabel('Net Asset Value')
ax.set_title('Global Asset Allocation - Net Asset Value Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. 结论

本研报复现了天风证券的半衰主成分风险平价模型(HPCRP)全球资产配置策略，主要结论：

1. **模型对比**: 对比了7种资产配置模型 (EW, EV, MV, MD, RP, PCRP, HPCRP)
2. **HPCRP表现**: 半衰主成分风险平价模型(HPCRP)通过引入短期动量效应，通常具有更好的收益风险比
3. **分散化**: PCRP和HPCRP通过对主成分进行风险均衡，能更好地分散化风险

---